# 2. When does the offline metric lie?

Compares AUC with incremental impact on a held-out test set.

In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from data import load_criteo, FEATURES

OUTCOME = "conversion"
SEED = 1

df = load_criteo()

strata = df["treatment"].astype(str) + df[OUTCOME].astype(str)
train, test = train_test_split(df, test_size=0.2, random_state=SEED, stratify=strata)

PARAMS = dict(
    n_estimators=150,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=200,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.9,
    random_state=SEED,
    verbose=-1,
)

outcome_model = lgb.LGBMClassifier(**PARAMS)
outcome_model.fit(train[FEATURES], train[OUTCOME])

outcome_score = outcome_model.predict_proba(test[FEATURES])[:, 1]
print(f"AUC: {roc_auc_score(test[OUTCOME], outcome_score):.3f}")